In [ ]:
!pip install -qq pydantic-ai serpapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.8/210.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.5/128.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.2/473.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [ ]:
from openai import AsyncAzureOpenAI
from google.colab import userdata

from pprint import pprint
import json
import httpx

import nest_asyncio

from dataclasses import dataclass
from IPython.display import Markdown, display, clear_output

import numpy as np

from typing import Literal, Union

from rich.prompt import Prompt
from rich import print
from pydantic_ai import Agent, Tool
from pydantic_ai.models.openai import OpenAIModel
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.messages import ModelMessage
from pydantic_ai.result import Usage, UsageLimits

import serpapi

In [ ]:
async_azure_client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))
nest_asyncio.apply()
model = OpenAIModel('gpt-4o', openai_client=async_azure_client)

In [ ]:
from IPython.display import Markdown, display

def printmd(string:str):
    display(Markdown(string))

In [ ]:
joke_selection_agent = Agent(
    model=model,
    system_prompt=(
        'Use the `joke_factory` to generate some jokes, then choose the best. '
        'You must return just a single joke.'
    ),
)
joke_generation_agent = Agent(model=model, result_type=list[str],
                              system_prompt=(
        'Generate the exact number of items in a list of strings.'
    ),
                              )


@joke_selection_agent.tool
async def joke_factory(ctx: RunContext[None], count: int) -> list[str]:
    print('Invoking `joke_factory` tool...')
    r = await joke_generation_agent.run(
        f'Please generate {count} jokes.',
        usage=ctx.usage,
    )
    return r.data


result = joke_selection_agent.run_sync(
    'Tell me a joke.',
    usage_limits=UsageLimits(request_limit=5, total_tokens_limit=1500),
)
print(result.data)
print(result.usage())

Invoking `joke_factory` tool...

Why don't scientists trust atoms? Because they make up everything!

Usage(
    requests=3,
    request_tokens=246,
    response_tokens=184,
    total_tokens=430,
    details={
        'accepted_prediction_tokens': 0,
        'audio_tokens': 0,
        'reasoning_tokens': 0,
        'rejected_prediction_tokens': 0,
        'cached_tokens': 0
    }
)

In [ ]:
result.new_messages()[-2].parts[0].content = 'Retry again'

In [ ]:
pprint(result.all_messages())

[ModelRequest(parts=[SystemPromptPart(content='Use the `joke_factory` to '
                                              'generate some jokes, then '
                                              'choose the best. You must '
                                              'return just a single joke.',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Tell me a joke.',
                                    timestamp=datetime.datetime(2025, 3, 3, 14, 10, 15, 623385, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='joke_factory',
                                   args='{"count":5}',
                                   tool_call_id='call_xVZqSSqRQI2D2bbUd4XRMB93',
                                   part_kind='tool-call')],
               model_name='gpt-4o-20

In [ ]:
@dataclass
class ClientAndKey:
    http_client: httpx.AsyncClient
    api_key: str


joke_selection_agent = Agent(
    model=model,
    deps_type=ClientAndKey,
    system_prompt=(
        'Use the `joke_factory` tool to generate some jokes on the given subject, '
        'then choose the best. You must return just a single joke.'
    ),
)
joke_generation_agent = Agent(
    model=model,
    deps_type=ClientAndKey,
    result_type=list[str],
    system_prompt=(
        'Use the "get_jokes" tool to get several jokes on the given subject in different messages, '
        'then extract each joke into a list.'
    ),
)


@joke_selection_agent.tool
async def joke_factory(ctx: RunContext[ClientAndKey], count: int) -> list[str]:
    print('Invoking `joke_factory` tool...')
    r = await joke_generation_agent.run(
        f'Please generate {count} jokes.',
        deps=ctx.deps,
        usage=ctx.usage,
    )
    print('Got response from `joke_factory` tool...')
    return r.data


@joke_generation_agent.tool
async def get_jokes(ctx: RunContext[ClientAndKey], count: int) -> str:
    print('Invoking `get_jokes` tool...')
    response = await ctx.deps.http_client.get(
        'https://example.com',
        params={'count': count},
        headers={'Authorization': f'Bearer {ctx.deps.api_key}'},
    )
    print('Got response from `get_jokes` tool...')
    response.raise_for_status()
    return response.text


async with httpx.AsyncClient() as client:
    deps = ClientAndKey(client, 'foobar')
    result = await joke_selection_agent.run('Tell me a joke.', deps=deps)
    print(result.data)
    print(result.usage())
    pprint(result.all_messages())

Invoking `joke_factory` tool...

Invoking `get_jokes` tool...

Got response from `get_jokes` tool...

Invoking `get_jokes` tool...

Got response from `get_jokes` tool...

...

Got response from `get_jokes` tool...

Got response from `joke_factory` tool...

Why don’t scientists trust atoms? Because they make up everything!

Usage(
    requests=11,
    request_tokens=17644,
    response_tokens=236,
    total_tokens=17880,
    details={
        'accepted_prediction_tokens': 0,
        'audio_tokens': 0,
        'reasoning_tokens': 0,
        'rejected_prediction_tokens': 0,
        'cached_tokens': 0
    }
)

[ModelRequest(parts=[SystemPromptPart(content='Use the `joke_factory` tool to '
                                              'generate some jokes on the '
                                              'given subject, then choose the '
                                              'best. You must return just a '
                                              'single joke.',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Tell me a joke.',
                                    timestamp=datetime.datetime(2025, 3, 3, 14, 10, 27, 719964, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='joke_factory',
                                   args='{"count":1}',
                                   tool_call_id='call_xaRmyv3ohGKsucZCVvvjcy4a',
                     

In [ ]:
class FlightDetails(BaseModel):
    flight_number: str


class Failed(BaseModel):
    """Unable to find a satisfactory choice."""


flight_search_agent = Agent[None, Union[FlightDetails, Failed]](
    model=model,
    result_type=Union[FlightDetails, Failed],  # type: ignore
    system_prompt=(
        'Use the "flight_search" tool to find a flight '
        'from the given origin to the given destination.'
    ),
)


@flight_search_agent.tool
async def flight_search(
    ctx: RunContext[None], origin: str, destination: str
) -> Union[FlightDetails, None]:
    # in reality, this would call a flight search API or
    # use a browser to scrape a flight search website
    return FlightDetails(flight_number='AK456')


usage_limits = UsageLimits(request_limit=15)


async def find_flight(usage: Usage) -> Union[FlightDetails, None]:
    message_history: Union[list[ModelMessage], None] = None
    for _ in range(3):
        prompt = Prompt.ask(
            'Where would you like to fly from and to?',
        )
        result = await flight_search_agent.run(
            prompt,
            message_history=message_history,
            usage=usage,
            usage_limits=usage_limits,
        )
        if isinstance(result.data, FlightDetails):
            return result.data
        else:
            result.new_messages()[-2].parts[0].content = 'Please Retry again'
            # message_history = result.all_messages(
            #     result_tool_return_content='Please try again.'
            # )


class SeatPreference(BaseModel):
    row: int = Field(ge=1, le=30)
    seat: Literal['A', 'B', 'C', 'D', 'E', 'F']


# This agent is responsible for extracting the user's seat selection
seat_preference_agent = Agent[None, Union[SeatPreference, Failed]](
    model=model,
    result_type=Union[SeatPreference, Failed],  # type: ignore
    system_prompt=(
        "Extract the user's seat preference. "
        'Seats A and F are window seats. '
        'Row 1 is the front row and has extra leg room. '
        'Rows 14, and 20 also have extra leg room. '
    ),
)


async def find_seat(usage: Usage) -> SeatPreference:
    message_history: Union[list[ModelMessage], None] = None
    while True:
        answer = Prompt.ask('What seat would you like?')

        result = await seat_preference_agent.run(
            answer,
            message_history=message_history,
            usage=usage,
            usage_limits=usage_limits,
        )
        if isinstance(result.data, SeatPreference):
            return result.data
        else:
            print('Could not understand seat preference. Please try again.')
            message_history = result.all_messages()


usage: Usage = Usage()

opt_flight_details = await find_flight(usage)
if opt_flight_details is not None:
    print(f'Flight found: {opt_flight_details.flight_number}')
    seat_preference = await find_seat(usage)
    print(f'Seat preference: {seat_preference}')

Where would you like to fly from and to?:

Lima


Flight found: AK456

What seat would you like?:

cuzco


Seat preference: row=1 seat='A'

In [ ]:
opt_flight_details = await find_flight(usage)
if opt_flight_details is not None:
    print(f'Flight found: {opt_flight_details.flight_number}')
    seat_preference = await find_seat(usage)
    print(f'Seat preference: {seat_preference}')

Where would you like to fly from and to?:

In [ ]:
usage

Usage(requests=11, request_tokens=1757, response_tokens=179, total_tokens=1936, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'cached_tokens': 0})

In [ ]:
opt_flight_details = await find_flight(usage)
if opt_flight_details is not None:
    print(f'Flight found: {opt_flight_details.flight_number}')
    seat_preference = await find_seat(usage)
    print(f'Seat preference: {seat_preference}')

Where would you like to fly from and to?:

from lima to callao
Flight found: AK456


What seat would you like?:

z
Seat preference: row=1 seat='A'


In [ ]:
result.all_messages(result_tool_return_content=...)

TypeError: _BaseRunResult.all_messages() got an unexpected keyword argument 'result_tool_return_content'

# ECHO


In [ ]:
agent_planner = Agent(model=model, result_type=str, system_prompt="generate some paragraph with ideas given the user topic, then generate 5 insights questions and answer them")
agent1 = Agent(model=model, result_type=str, system_prompt="generate a useful paragraph with ideas given a topic")
agent2 = Agent(model=model, result_type=str, system_prompt="generate useful insight question")
agent3 = Agent(model=model, result_type=str, system_prompt="Answer in one sentence the insight question")
    # r = await async_azure_client.chat.completions.create(messages=[{'role':'user','content':f'Generate a paragraph with ideas about `{topic}`'}], model='gpt-4o')

@agent_planner.tool
async def generate_a_useful_paragraph(ctx: RunContext[None], topic: str) -> str:
    print('Invoking `generate_a_useful_paragraph` tool...')
    return agent1.run_sync(topic).data

@agent_planner.tool
async def generate_a_question(ctx: RunContext[None], sentence: str) -> str:
    print('Invoking `generate_a_question` tool...')
    return agent2.run_sync(sentence).data

@agent_planner.tool
async def answer_a_question(ctx: RunContext[None], question: str) -> str:
    print('Invoking `answer_a_question` tool...')
    return agent3.run_sync(question).data

result = agent_planner.run_sync('machine learning')

Invoking `generate_a_useful_paragraph` tool...

Invoking `generate_a_question` tool...

Invoking `generate_a_question` tool...

Invoking `generate_a_question` tool...

Invoking `generate_a_question` tool...

...

Invoking `answer_a_question` tool...

Invoking `answer_a_question` tool...

Invoking `answer_a_question` tool...

Invoking `answer_a_question` tool...

Invoking `answer_a_question` tool...

In [ ]:
print(result.data)

**Insights Questions and Answers**

1. **What are the latest advancements in machine learning algorithms and how are they improving the accuracy and 
efficiency of predictions and decisions in various industries?**
   - The latest advancements in machine learning algorithms, including transformer-based models, improved neural 
network architectures, and more efficient optimization techniques, are significantly enhancing the accuracy and 
efficiency of predictions and decisions across various industries by enabling better data processing, more precise 
pattern recognition, and faster computational speeds.

2. **What are the latest advancements in machine learning algorithms that have significantly impacted the 
effectiveness of natural language processing, image recognition, recommendation systems, and predictive 
analytics?**
   - The latest advancements in machine learning algorithms that have significantly impacted effectiveness across 
natural language processing, image recognition, recommendation systems, and predictive analytics include 
transformer-based models like GPT-4, Vision Transformers (ViTs), graph neural networks, and reinforcement learning 
frameworks.

3. **How can organizations effectively determine which type of machine learning technique (supervised, 
unsupervised, or reinforcement learning) is best suited for addressing specific business challenges or project 
goals?**
   - Organizations can effectively determine the most suitable machine learning technique for specific business 
challenges by evaluating key factors such as the nature of the data (labeled or unlabeled), the desired outcomes 
(predictions, data insights, or improved decision-making), and the actionability of results, aligning these with 
the capabilities and characteristics of supervised, unsupervised, or reinforcement learning methods.

4. **What are the key ethical considerations that companies must address when implementing machine learning 
solutions in sensitive industries such as healthcare and finance?**
   - Key ethical considerations include ensuring data privacy and security, mitigating biases, maintaining 
transparency, obtaining informed consent, and ensuring accountability in decision-making processes.

5. **What are the key challenges and barriers businesses and researchers face when implementing machine learning 
solutions, and how can they be effectively addressed to maximize the impact of these technologies?**
   - Key challenges in implementing machine learning solutions include data quality and availability, algorithm 
bias, high computational costs, and a lack of skilled personnel, which can be addressed by ensuring robust data 
management practices, employing diverse teams to mitigate bias, investing in scalable infrastructure, and providing
continuous education and training for the workforce.

In [ ]:
print(result.all_messages() )

[
    ModelRequest(
        parts=[
            SystemPromptPart(
                content='generate some paragraph with ideas given the user topic, then generate 5 insights 
questions and answer them',
                dynamic_ref=None,
                part_kind='system-prompt'
            ),
            UserPromptPart(
                content='machine learning',
                timestamp=datetime.datetime(2025, 3, 3, 14, 11, 53, 891798, tzinfo=datetime.timezone.utc),
                part_kind='user-prompt'
            )
        ],
        kind='request'
    ),
    ModelResponse(
        parts=[
            ToolCallPart(
                tool_name='generate_a_useful_paragraph',
                args='{"topic":"machine learning"}',
                tool_call_id='call_vqQ7NfXvMyvB2SedJrCFEl40',
                part_kind='tool-call'
            )
        ],
        model_name='gpt-4o-2024-05-13',
        timestamp=datetime.datetime(2025, 3, 3, 14, 11, 54, tzinfo=datetime.timezone.utc),
        kind='response'
    ),
    ModelRequest(
        parts=[
            ToolReturnPart(
                tool_name='generate_a_useful_paragraph',
                content='Machine learning is a rapidly evolving field that leverages algorithms and statistical 
models to enable computers to learn from and make predictions or decisions based on data. It spans various 
applications, from natural language processing and image recognition to recommendation systems and predictive 
analytics. By utilizing techniques such as supervised, unsupervised, and reinforcement learning, machine learning 
systems can identify patterns and improve their performance over time, often without explicit programming. As the 
volume of data continues to grow and computational power advances, machine learning holds the promise of driving 
significant innovations across numerous industries, including healthcare, finance, autonomous vehicles, and 
cybersecurity. Businesses and researchers are increasingly embracing machine learning to gain insights, optimize 
operations, and develop smarter, more adaptive technologies.',
                tool_call_id='call_vqQ7NfXvMyvB2SedJrCFEl40',
                timestamp=datetime.datetime(2025, 3, 3, 14, 11, 56, 434272, tzinfo=datetime.timezone.utc),
                part_kind='tool-return'
            )
        ],
        kind='request'
    ),
    ModelResponse(
        parts=[
            ToolCallPart(
                tool_name='generate_a_question',
                args='{"sentence": "Machine learning is a rapidly evolving field that leverages algorithms and 
statistical models to enable computers to learn from and make predictions or decisions based on data."}',
                tool_call_id='call_6logbsFfQIS2y5Xt7Y3DHFuV',
                part_kind='tool-call'
            ),
            ToolCallPart(
                tool_name='generate_a_question',
                args='{"sentence": "Machine learning spans various applications, from natural language processing 
and image recognition to recommendation systems and predictive analytics."}',
                tool_call_id='call_ymQnPkQoaUEUgnNPoenHiybt',
                part_kind='tool-call'
            ),
            ToolCallPart(
                tool_name='generate_a_question',
                args='{"sentence": "By utilizing techniques such as supervised, unsupervised, and reinforcement 
learning, machine learning systems can identify patterns and improve their performance over time, often without 
explicit programming."}',
                tool_call_id='call_iVXGlfZbxxItoYzadxq0V8S8',
                part_kind='tool-call'
            ),
            ToolCallPart(
                tool_name='generate_a_question',
                args='{"sentence": "Machine learning holds the promise of driving significant innovations across 
numerous industries, including healthcare, finance, autonomous vehicles, and cybersecurity."}',
                tool_call_id='call_ozts65lx7t7t5Fu0b5qZbj2U',
               

In [ ]:
class ParagraphResult(BaseModel):
    paragraph: str = Field(description='A generated paragraph')
class InsightResult(BaseModel):
    question: str = Field(description='A insight question')
    pregunta: str = Field(description='the generated insight question in spanish')

agent_planner = Agent(model=model, result_type=str, system_prompt="Given the user prompt and a number, you must generate a plan to use the avilable tools to use a paragraph, generate and answer insights(length is provided by user)")
agent1 = Agent(model=model, result_type=ParagraphResult, system_prompt="generate a useful paragraph with ideas given a topic")
agent2 = Agent(model=model, result_type=InsightResult, system_prompt="generate useful insight question")
agent3 = Agent(model=model, result_type=list[str], system_prompt="Answer in one sentence the insight question")

@agent_planner.tool_plain
async def generate_a_useful_paragraph(topic: str) -> ParagraphResult:
    print('Invoking `generate_a_useful_paragraph` tool...')
    return agent1.run_sync(topic).data

@agent_planner.tool_plain
async def generate_a_question(sentence: str) -> InsightResult:
    print('Invoking `generate_a_question` tool...')
    return agent2.run_sync(sentence).data

@agent_planner.tool_plain
async def answer_a_question(insight: InsightResult) -> str:
    print('Invoking `answer_a_question` tool...')
    return '\n\n'.join(agent3.run_sync(insight.pregunta).data)

result = agent_planner.run_sync('machine learning and 2 insights' )

Invoking `generate_a_useful_paragraph` tool...

Invoking `generate_a_question` tool...

Invoking `generate_a_question` tool...

Invoking `answer_a_question` tool...

Invoking `answer_a_question` tool...

In [ ]:
printmd(result.data)

Here is a paragraph on machine learning along with two insights based on the paragraph:

### Paragraph:
Machine learning is a transformative technology that involves the development of algorithms and statistical models that enable computers to perform tasks without explicit instruction. Instead, these systems learn from data, identifying patterns, making predictions, and improving performance over time. With applications ranging from autonomous vehicles and personalized recommendations to fraud detection and healthcare diagnostics, machine learning is revolutionizing many industries. Its ability to handle vast amounts of data and derive meaningful insights is one of its most powerful features, making it an essential tool in the era of big data and artificial intelligence.

### Insights:

1. **Challenges and Limitations in Implementing Machine Learning:**
   Organizations face several challenges and limitations when implementing machine learning technologies, including the lack of quality data, the need for specialized talent, biases in models, high costs, and the technological infrastructure necessary for deployment.

2. **Revolutionizing Industries:**
   In the industry of autonomous vehicles, machine learning enables cars to drive autonomously by interpreting real-time data from their sensors and cameras. This technology is also revolutionizing other industries such as personalized recommendations, fraud detection, and healthcare diagnostics.

In [ ]:
# pprint(result.all_messages(), width=160)
print(result.all_messages())

[
    ModelRequest(
        parts=[
            SystemPromptPart(
                content='Given the user prompt and a number, you must generate a plan to use the avilable tools to 
use a paragraph, generate and answer insights(length is provided by user)',
                dynamic_ref=None,
                part_kind='system-prompt'
            ),
            UserPromptPart(
                content='machine learning and 2 insights',
                timestamp=datetime.datetime(2025, 3, 3, 14, 13, 3, 398602, tzinfo=datetime.timezone.utc),
                part_kind='user-prompt'
            )
        ],
        kind='request'
    ),
    ModelResponse(
        parts=[
            ToolCallPart(
                tool_name='generate_a_useful_paragraph',
                args='{"topic":"machine learning"}',
                tool_call_id='call_dd6cSyDQDxrWHPYoL2QFUM0J',
                part_kind='tool-call'
            )
        ],
        model_name='gpt-4o-2024-05-13',
        timestamp=datetime.datetime(2025, 3, 3, 14, 13, 3, tzinfo=datetime.timezone.utc),
        kind='response'
    ),
    ModelRequest(
        parts=[
            ToolReturnPart(
                tool_name='generate_a_useful_paragraph',
                content=ParagraphResult(
                    paragraph='Machine learning is a transformative technology that involves the development of 
algorithms and statistical models that enable computers to perform tasks without explicit instruction. Instead, 
these systems learn from data, identifying patterns, making predictions, and improving performance over time. With 
applications ranging from autonomous vehicles and personalized recommendations to fraud detection and healthcare 
diagnostics, machine learning is revolutionizing many industries. Its ability to handle vast amounts of data and 
derive meaningful insights is one of its most powerful features, making it an essential tool in the era of big data
and artificial intelligence.'
                ),
                tool_call_id='call_dd6cSyDQDxrWHPYoL2QFUM0J',
                timestamp=datetime.datetime(2025, 3, 3, 14, 13, 5, 660654, tzinfo=datetime.timezone.utc),
                part_kind='tool-return'
            )
        ],
        kind='request'
    ),
    ModelResponse(
        parts=[
            ToolCallPart(
                tool_name='generate_a_question',
                args='{"sentence": "Machine learning is a transformative technology that involves the development 
of algorithms and statistical models that enable computers to perform tasks without explicit instruction."}',
                tool_call_id='call_oGWKgIdBCeI1pZLO7rwmFH4H',
                part_kind='tool-call'
            ),
            ToolCallPart(
                tool_name='generate_a_question',
                args='{"sentence": "With applications ranging from autonomous vehicles and personalized 
recommendations to fraud detection and healthcare diagnostics, machine learning is revolutionizing many 
industries."}',
                tool_call_id='call_GVWxOjlcdCWqufZMOblifwFA',
                part_kind='tool-call'
            )
        ],
        model_name='gpt-4o-2024-05-13',
        timestamp=datetime.datetime(2025, 3, 3, 14, 13, 5, tzinfo=datetime.timezone.utc),
        kind='response'
    ),
    ModelRequest(
        parts=[
            ToolReturnPart(
                tool_name='generate_a_question',
                content=InsightResult(
                    question='What challenges and limitations do organizations face when implementing machine 
learning technologies?',
                    pregunta='¿Qué desafíos y limitaciones enfrentan las organizaciones al implementar tecnologías 
de aprendizaje automático?'
                ),
                tool_call_id='call_oGWKgIdBCeI1pZLO7rwmFH4H',
                timestamp=datetime.datetime(2025, 3, 3, 14, 13, 8, 931333, tzinfo=datetime.timezone.utc),
                part_kind='tool-return'
            ),
            ToolReturnPart(
           

# SERPAPI

In [ ]:
import time
time.timezone

0

In [ ]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

def generate_date_frame(**kwargs)->str:
    kwargs = kwargs or {"months":1}
    end_date = datetime.now()
    start_date = end_date - relativedelta(**kwargs)
    return f"{start_date.strftime('%Y-%m-%d')}..{end_date.strftime('%Y-%m-%d')}"

generate_date_frame(months=3)

'2024-11-05..2025-02-05'

In [ ]:
client = serpapi.Client(api_key=userdata.get('SERPAPI_API_KEY'))
duckduckgo_results = client.search({
    'engine': 'duckduckgo',
    'q': 'What happens with Elon Musk Tasla',
    'df': generate_date_frame(days=3),
})

duckduckgo_results

{
    "search_metadata": {
        "id": "67a38ac1fc9f120207f78f1e",
        "status": "Success",
        "json_endpoint": "https://serpapi.com/searches/e0984f95e76ee477/67a38ac1fc9f120207f78f1e.json",
        "created_at": "2025-02-05 15:58:57 UTC",
        "processed_at": "2025-02-05 15:58:57 UTC",
        "duckduckgo_url": "https://duckduckgo.com/?q=What+happens+with+Elon+Musk+Tasla&kl=us-en&df=2025-02-02..2025-02-05",
        "raw_html_file": "https://serpapi.com/searches/e0984f95e76ee477/67a38ac1fc9f120207f78f1e.html",
        "prettify_html_file": "https://serpapi.com/searches/e0984f95e76ee477/67a38ac1fc9f120207f78f1e.prettify",
        "total_time_taken": 1.43
    },
    "search_parameters": {
        "engine": "duckduckgo",
        "q": "What happens with Elon Musk Tasla",
        "kl": "us-en",
        "df": "2025-02-02..2025-02-05"
    },
    "search_information": {
        "organic_results_state": "Including results for spelling fix",
        "spelling_fix": "what happens wi

In [ ]:
context_snippets = list(map(lambda x: f"snippet:{x['snippet']},source:{x['link']}" ,duckduckgo_results['organic_results']))
context = '\n\n'.join(context_snippets)
print(context)

snippet:There finally appears to be some Tesla shareholder momentum to fire Elon Musk from the company after years of concerns being ignored by the board and most shareholders.,source:https://electrek.co/2025/02/04/theres-finally-some-tesla-tsla-shareholder-momentum-to-fire-elon-musk/

snippet:In Washington these days, Elon Musk seems to be everywhere.In the 15 days Donald Trump has been back in the White House, Musk and his Department of Government Efficiency have been moving to change ...,source:https://www.npr.org/2025/02/04/1228912524/what-does-elon-musk-get-out-of-remaking-the-government-with-doge

snippet:Elon Musk's control of a federal payment system is raising concerns about funding for programs like Social Security and the safety of personal data. ... Musk, the billionaire CEO of Tesla, gained ...,source:https://www.cbsnews.com/news/musk-treasury-social-security-access-federal-payment-system-trump/

snippet:The dip knocked $11.8 billion off of the net worth of Musk, according

In [ ]:
client = serpapi.Client(api_key=userdata.get('SERPAPI_API_KEY'))
duckduckgo_news_results = client.search({
    'engine': 'duckduckgo_news',
    'q': 'What happens with Elon Musk and Tesla',
    'df': generate_date_frame(days=3),
})

duckduckgo_news_results

{
    "search_metadata": {
        "id": "67a38adf93180831f6580693",
        "status": "Success",
        "json_endpoint": "https://serpapi.com/searches/98a9fe1dbd90fb18/67a38adf93180831f6580693.json",
        "created_at": "2025-02-05 15:59:27 UTC",
        "processed_at": "2025-02-05 15:59:27 UTC",
        "duckduckgo_news_url": "https://duckduckgo.com/?q=What+happens+with+Elon+Musk+and+Tesla&kl=us-en&df=2025-02-02..2025-02-05&iar=news&ia=news",
        "raw_html_file": "https://serpapi.com/searches/98a9fe1dbd90fb18/67a38adf93180831f6580693.html",
        "prettify_html_file": "https://serpapi.com/searches/98a9fe1dbd90fb18/67a38adf93180831f6580693.prettify",
        "total_time_taken": 1.69
    },
    "search_parameters": {
        "engine": "duckduckgo_news",
        "q": "What happens with Elon Musk and Tesla",
        "kl": "us-en",
        "df": "2025-02-02..2025-02-05"
    },
    "search_information": {
        "news_results_state": "Results for exact spelling"
    },
    "news_

In [ ]:
context_snippets = list(
    map(
        lambda x: f"{x['snippet']}\n\n<a href='{x.get('link')}'><img src='{x.get('thumbnail')}' alt='isolated' width='300'/></a>",
        filter(lambda x: 'thumbnail' in x and x['thumbnail'], duckduckgo_news_results['news_results'])
    )
)
context = f'\n\n{("-"*3)}\n\n'.join(context_snippets)

In [ ]:
printmd(context)

While Tesla's sales in markets such as Belgium, Netherlands, and Sweden rose last year, the overall picture in Europe is less than rosy for the automaker — esp

<a href='https://www.msn.com/en-us/money/technology/tesla-sales-in-europe-are-sliding-that-s-a-problem-for-elon-musk/ar-AA1xNnDB'><img src='https://img-s-msn-com.akamaized.net/tenant/amp/entityid/AA1xNgcG.img?w=3966&h=2974&m=4&q=75' alt='isolated' width='300'/></a>

---

Tesla (NASDAQ: TSLA) shares initially shrugged off a weak earnings report before the stock got caught in the trade-war sell-off. The electric vehicle (EV) maker had a tough year in 2024, although Chief Executive Officer Elon Musk continues to be very upbeat about the future.

<a href='https://www.msn.com/en-us/money/companies/ceo-elon-musk-says-tesla-could-be-worth-more-than-the-worlds-5-largest-companies-is-the-stock-a-buy/ar-AA1ys3Y6'><img src='https://img-s-msn-com.akamaized.net/tenant/amp/entityid/AA1ysd48.img?w=4206&h=2804&m=4&q=80' alt='isolated' width='300'/></a>

---

Elon Musk had a lot to say about Tesla's upcoming Robotaxi service on Wednesday. Robotaxi will be coming to Austin and other cities later this year, Musk claims.

<a href='https://www.forbes.com/sites/brookecrothers/2025/02/02/how-tesla-robotaxi-will-work-according-to-elon-musk-kind-of-like-airbnb/'><img src='https://imageio.forbes.com/specials-images/imageserve/679f9ea86620a1ddd3aa97fa/0x0.jpg?format=jpg&height=900&width=1600&fit=bounds' alt='isolated' width='300'/></a>

---

In fact, Musk thinks humanoid robots will outnumber humans by 2040. He believes Tesla will manufacture them at a cost of around $20,000, which opens the door to a sale price of under $30,000. That price will probably decline over time as the company becomes more efficient, and as competitors enter the market.

<a href='https://www.msn.com/en-us/money/companies/elon-musk-predicts-tesla-will-be-bigger-than-apple-nvidia-microsoft-amazon-and-alphabet-combined-thanks-to-this-10-trillion-opportunity/ar-AA1ygQ56'><img src='https://g.foolcdn.com/editorial/images/805839/a-tesla-dealership-with-two-tesla-electric-vehicles-parked-out-front.png' alt='isolated' width='300'/></a>

---

It doesn't matter what Musk really meant with this salute. Robotaxis are exclusively a product for large urban areas, and that population largely didn't like it at all.

<a href='https://www.forbes.com/sites/bradtempleton/2025/01/27/did-elon-musks-salute-cripple-the-tesla-robotaxi/'><img src='https://imageio.forbes.com/specials-images/imageserve/67956dc5a48768fb935cb4e1/0x0.jpg?format=jpg&height=900&width=1600&fit=bounds' alt='isolated' width='300'/></a>

---

Elon Musk finally admits that Tesla will have to replace its HW3 self-driving computers. He said it would be difficult, but Tesla would do it. However, no concrete plan has been shared. For the better part of the year, we have been reporting that Tesla can't achieve its promise of "full self-driving" on HW3, and it needs to come clean about it.

<a href='https://electrek.co/2025/01/29/elon-musk-finally-admits-that-tesla-will-have-to-replace-its-hw3-self-driving-computers/'><img src='https://electrek.co/wp-content/uploads/sites/3/2019/04/Screen-Shot-2019-04-22-at-2.48.54-PM-e1568302212821.jpg?quality=82&strip=all&w=1600' alt='isolated' width='300'/></a>

---

Tesla's driver assistance software, known as full self-driving, or FSD, will see unsupervised tests in Texas, California and other states this year.

<a href='https://www.msn.com/en-us/autos/news/tesla-sales-disappoint-but-elon-musk-vows-new-cars-will-be-in-the-wild-with-no-one-in-them/ar-AA1y5NPG'><img src='https://nypost.com/wp-content/uploads/sites/2/2025/01/2025-citing-declining-vehicle-prices-97573174.jpg?w=1024' alt='isolated' width='300'/></a>

---

Tesla sold almost 60% fewer cars in Germany in January than in the year-earlier period, as the U.S. electric vehicle maker faces a test of popularity amid headline-grabbing political involvement by its billionaire CEO Elon Musk.

<a href='https://www.reuters.com/business/autos-transportation/tesla-sees-german-car-sales-plunge-january-2025-02-05/'><img src='https://www.reuters.com/resizer/v2/57REII25SNLDJEYRWREAFTGLBI.jpg?auth=c1c24ed0f1cc70fa11a0dc71dcddeb1dad7ea6b9844bedf04e74991cf490c7bb&height=1005&width=1920&quality=80&smart=true' alt='isolated' width='300'/></a>

---

Elon Musk said Austin residents will be able to pay for a fully autonomous Tesla robotaxi ride in June, with an expansion to more US cities planned.

<a href='https://www.msn.com/en-us/autos/news/elon-musk-says-a-paid-robotaxi-service-launches-in-june-and-tesla-will-dip-a-toe-in-the-water-before-expanding-it/ar-AA1y8ONa'><img src='https://i.insider.com/679bc661196626c40985d4c8?width=1200&format=jpeg' alt='isolated' width='300'/></a>

---

Congress is proving little match for the Department of Government Efficiency as wary lawmakers watch it march through the bureaucracy.

<a href='https://www.msn.com/en-us/news/politics/trump-and-musks-dismantling-of-government-is-shaking-the-foundations-of-us-democracy/ar-AA1ysjZA'><img src='https://dims.apnews.com/dims4/default/9b5d9bd/2147483647/strip/true/crop/5652x3179+0+294/resize/1440x810!/quality/90/?url=https%3A%2F%2Fassets.apnews.com%2F58%2F7d%2F2f6dfa75e820e887756123a5e9f0%2Ff628a3db342e445b98c6f04b3f277d40' alt='isolated' width='300'/></a>

---

Elon Musk said Tesla will begin launching unsupervised self-driving models in Austin, Texas by June and several other U.S. cities by the end of 2025.

<a href='https://www.msn.com/en-ca/autos/news/tesla-earnings-call-elon-musk-reveals-timeline-for-driverless-cars-humanoid-robots/ar-AA1y5M3D'><img src='https://www.usatoday.com/gcdn/authoring/authoring-images/2025/01/30/USAT/78036558007-usatsi-25278482.jpg?crop=629,354,x0,y61&width=629&height=354&format=pjpg&auto=webp' alt='isolated' width='300'/></a>

---

According to two federal employees, DOGE representatives didn't provide their last names during recent interactions with workers.

<a href='https://www.msn.com/en-us/news/technology/some-members-of-elon-musks-doge-squad-arent-sharing-their-last-names-as-they-attempt-to-remake-the-federal-workforce/ar-AA1ypZks'><img src='https://i.insider.com/67a28c61eb4be2fff9a3809e?width=1200&format=jpeg' alt='isolated' width='300'/></a>

---

Elon Musk promised Tesla's car sales would jump 20% this year at a minimum, but his own company doesn't even seem to believe that.

<a href='https://www.msn.com/en-us/news/technology/a-peak-under-the-hood-of-elon-musk-s-tesla-reveals-a-worrying-trend-its-auto-business-is-rusting-away/ar-AA1y8Lpk'><img src='https://s.yimg.com/ny/api/res/1.2/Wf6mf91HFVmXqHYZaikkZA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD04MDM-/https://media.zenfs.com/en/aol_fortune_385/b120ee4eeb9d992f52fbd138f6a4fb7e' alt='isolated' width='300'/></a>

---

The SGE designation, which has come under scrutiny under Democrats as well, exempts short-term federal employees from certain disclosure rules.

<a href='https://www.msn.com/en-us/politics/government/elon-musk-is-a-special-government-employee-what-does-that-mean/ar-AA1yohXF'><img src='https://www.washingtonpost.com/wp-apps/imrs.php?src=https://arc-anglerfish-washpost-prod-washpost.s3.amazonaws.com/public/IID7ETYNV4VW4QBCC5ZAW3HVZM_size-normalized.jpg&w=1440' alt='isolated' width='300'/></a>

---

Astronomers have retracted the discovery of a new asteroid after realizing the object was the remains of Elon Musk's Tesla Roadster and its driver "Starman," which were launched into space in 2018.

<a href='https://www.msn.com/en-us/news/technology/newly-discovered-near-earth-asteroid-isnt-an-asteroid-at-all-its-elon-musks-trashed-tesla/ar-AA1y8FLT'><img src='https://cdn.mos.cms.futurecdn.net/uYDYyWe8M2eNmN7DSKaWik-1200-80.jpg' alt='isolated' width='300'/></a>

---

Musk, who is now the head of the US Department of Government Efficiency (DOGE), may push Modi for a level-playing field for Tesla in India.

<a href='https://www.cnbctv18.com/world/teslas-elon-musk-likely-to-call-on-pm-narendra-modi-next-week-19552983.htm'><img src='https://images.cnbctv18.com/uploads/2023/06/pm-modi-musk-tesla-2015-san-jose-official-twitter-account-of-pm-modi.jpg?im=FitAndFill,width=500,height=300' alt='isolated' width='300'/></a>

---

Tesla (TSLA) stock gained 4.1% in yesterday's after-hours trading session fueled by CEO Elon Musk's comments that the vehicle business would

<a href='https://markets.businessinsider.com/news/stocks/tesla-stock-tsla-soars-on-elon-musk-s-optimistic-2025-growth-outlook-1034287276'><img src='https://blog.tipranks.com/wp-content/uploads/2025/01/30.1_2.png?utm_source=markets.businessinsider.com&utm_medium=referral' alt='isolated' width='300'/></a>

---

Elon Musk has been named as a "special government employee" while overseeing DOGE, according to the White House.

<a href='https://www.msn.com/en-us/politics/government/elon-musk-serving-as-special-government-employee-what-does-that-mean/ar-AA1yoFTX'><img src='https://s.yimg.com/ny/api/res/1.2/AhGP4nGcKQdkfbb32wXFAg--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD02NzU-/https://media.zenfs.com/en/livenow_fox_840/2c89acd4ae8bdfc082de2f9042fc1ee8' alt='isolated' width='300'/></a>

---

It's hard to fault a CEO who grows a company beyond $1 trillion in value. Elon Musk managed the feat by upending the automotive market with Tesla's electric vehicles and extended its lead with broader battery power.

<a href='https://www.reuters.com/breakingviews/elon-musk-dangerously-overrides-teslas-autopilot-2025-01-29/'><img src='https://www.reuters.com/resizer/v2/SCRGH256FBIBXDME75DYYKNJPY.jpg?auth=aaa10cf138a422826dfa0a3c837e533d0e05e8a51f743bfc4bcde045f956dd13&height=1005&width=1920&quality=80&smart=true' alt='isolated' width='300'/></a>

---

White House spokeswoman Karoline Leavitt said that Tesla CEO Musk is categorized as a special government employee.

<a href='https://www.msn.com/en-us/money/news/what-is-elon-musks-role-as-a-special-government-employee/ar-AA1ypU9k'><img src='https://assets3.cbsnewsstatic.com/hub/i/r/2025/02/03/fe50c3cc-a32e-4dec-a1e3-1c1b62519a86/thumbnail/1200x630/a264274459250dc3122f48ee749a0a56/cbsn-fusion-elon-musk-says-trump-agreed-to-shut-down-usaid-thumbnail.jpg?v=aaeeb2bb1dd1cd7107e4d78154d17e02' alt='isolated' width='300'/></a>

---

Musk, who also runs SpaceX and a number of other companies, has been known to make big often exaggerated promises. For example, he once called the Hyperloop, which was never built, the "fifth mode of transport" and famously promised to take Tesla private in a Twitter post, saying he had secured financing that he didn't have.

<a href='https://www.msn.com/en-us/money/topstocks/elon-musk-just-said-tesla-could-10x-from-here-is-he-right/ar-AA1yo08d'><img src='https://g.foolcdn.com/editorial/images/805984/tesla-model-3.jpg' alt='isolated' width='300'/></a>

---

Financial writer warns against buying Tesla, Inc. stock, citing declining fundamentals and unrealistic promises from CEO Elon Musk. Click for this TSLA update.

<a href='https://seekingalpha.com/article/4754387-elon-musk-keeps-overpromising-while-tesla-keeps-underdelivering-sell'><img src='https://static.seekingalpha.com/cdn/s3/uploads/getty_images/1272634746/image_1272634746.jpg?io=getty-c-w1536' alt='isolated' width='300'/></a>

---

The news comes from CEO Elon Musk, who finally admitted it during Wednesday's Tesla earnings call (via Electrek ). "The truth is that we will need to replace all HW3 computers in vehicles where FSD was purchased," said Musk after Tesla's head of FSD, Ashok Elluswamy, said the company is "not giving up on it."

<a href='https://www.msn.com/en-us/autos/news/elon-musk-admits-that-tesla-will-have-to-replace-old-computers-for-fsd-buyers/ar-AA1y6Z6a'><img src='https://helios-i.mashable.com/imagery/articles/066NOl0IFF42wH52of3hbsT/hero-image.fill.size_1200x675.v1738229499.png' alt='isolated' width='300'/></a>

---

While the initial testing will use Tesla's own fleet of cars, regular customers may be able to add their cars to the ride-hailing fleet next year, Musk says.

<a href='https://www.msn.com/en-us/autos/news/elon-musk-tesla-robotaxis-are-coming-to-austin-in-june/ar-AA1y8kdY'><img src='https://media.zenfs.com/en/pc_mag_263/5d60a9e498e058e96df60de7f1d673fa' alt='isolated' width='300'/></a>

---

The wannabe asteroid, announced on Jan. 2 as 2018 CN41, is actually a Tesla Roadster launched into space years ago by SpaceX CEO Elon Musk. The company sent the car (with a spacesuit-clad mannequin called "Starman" in the driver's seat) into a long orbit around the sun in 2018 as the first payload of the company's Falcon Heavy rocket.

<a href='https://www.msn.com/en-us/news/technology/when-is-an-asteroid-not-an-asteroid-when-its-elon-musks-tesla-roadster-it-turns-out/ar-AA1ybjJ8'><img src='https://cdn.mos.cms.futurecdn.net/e2cqB2EkXWvGUg4euk4Fe8-320-80.jpg' alt='isolated' width='300'/></a>

---

This dude is probably one of the most unintelligent billionaires I have ever met or seen or witnessed," Rep. Alexandria Ocasio-Cortez said during an Instagram Live video session.

<a href='https://www.msn.com/en-us/politics/general/aoc-mocked-after-calling-elon-musk-one-of-the-most-unintelligent-billionaires-i-have-ever-met/ar-AA1ypBek'><img src='https://nypost.com/wp-content/uploads/sites/2/2025/02/newspress-collage-evfyq9ndn-1738699496298.jpg?quality=75&strip=all&1738681532&w=1024' alt='isolated' width='300'/></a>

In [ ]:
agent_planner = Agent(model=model, result_type=str,
                      system_prompt=(
                          "Given a topic, search for internet news, "
                          "then generate 4 useful and interesting insights/questions. "
                          "Answer each question in separate paragraphs, including references."
                      ))
class Serpapi:
    client = serpapi.Client(api_key=userdata.get('SERPAPI_API_KEY'))


    @classmethod
    async def search_topic_until_today(cls, *, question: str, date_frame:str) -> str | None:
        print(f"{question=}, {date_frame=}")
        duckduckgo_results = cls.client.duckduck_results = client.search({
            'engine': 'duckduckgo', # yahho-finance, arvix
            'q': question,
            'df': generate_date_frame(**json.loads(date_frame))
        })
        if duckduckgo_results:
            context_snippets = list(map(lambda x: f"snippet:{x['snippet']},source:{x['link']}" ,duckduckgo_results['organic_results']))
            return '\n\n'.join(context_snippets)


@dataclass
class SupportDependencies:
    serpapi_client: Serpapi


@agent_planner.tool
async def search_news(ctx: RunContext[SupportDependencies], topic: str, date_frame:str) -> str:
    """
    Args:
        date_frame: A limit of days,months,weeks before tday, format in json like {"days":1}, {"months":1}, {"weeks":1}, {"months":1, "weeks":1, "days":1}
    """
    return await ctx.deps.serpapi_client.search_topic_until_today(question=topic, date_frame=date_frame)

@agent_planner.tool
async def answer_an_insight_question(ctx: RunContext[SupportDependencies], insight_question: str) -> str:
    return await ctx.deps.serpapi_client.search_topic_until_today(question=insight_question, date_frame='{"weeks":1}')

deps = SupportDependencies(serpapi_client=Serpapi())
result = agent_planner.run_sync('OpenAI in the last two weeks', deps=deps, message_history=[])
printmd(result.data)

question='OpenAI', date_frame='{"weeks":2}'
question="What are the main features of OpenAI's newly launched model, o3-mini?", date_frame='{"weeks":1}'
question='What is the dispute involving Elon Musk and OpenAI about?', date_frame='{"weeks":1}'
question='How is OpenAI planning to leverage its AI tools for the public sector?', date_frame='{"weeks":1}'
question="What does OpenAI's new 'Deep Research' tool aim to achieve?", date_frame='{"weeks":1}'


### What are the main features of OpenAI's newly launched model, o3-mini?

OpenAI's newly launched o3-mini model primarily focuses on enhancing reasoning tasks while maintaining cost-effectiveness. It is an advanced model within OpenAI's reasoning series, boasting significantly lower running costs—63% less than its predecessor, o1-mini, per input token. Notable new features include a larger context window supporting up to 200,000 tokens and the ability to produce outputs with a maximum length of 100,000 tokens. The model also excels in STEM fields, particularly science, mathematics, and coding, and exhibits remarkable performance in generating structured outputs, function calls, and developer messages. It is available for ChatGPT users, including both free-tier users (with limited access) and subscribers of Plus, Team, and Pro tiers. The o3-mini model is designed to be production-ready with options for varying degrees of reasoning effort (low, medium, and high) to cater to diverse computational needs ([TechCrunch](https://techcrunch.com/2025/01/31/openai-launches-o3-mini-its-latest-reasoning-model/), [OpenAI](https://openai.com/index/openai-o3-mini/)).

### What is the dispute involving Elon Musk and OpenAI about?

The dispute between Elon Musk and OpenAI centers around Musk's legal attempts to block the non-profit's transformation into a for-profit entity. This conflict dates back to a power struggle in 2017 when Sam Altman became CEO of OpenAI. Musk's lawsuit claims that OpenAI manipulated him into supporting its for-profit restructuring—a move Musk views as a betrayal of the organization's original non-profit mission. He argues that this transition benefits OpenAI at his expense and his competing AI interests. Musk has requested a court order to prevent OpenAI's restructuring, alleging that it would disadvantage him and debilitate OpenAI's business and mission. The legal battles have included Musk's addition of new claims and defendants, aiming to halt the restructuring and safeguard his interests ([AP News](https://apnews.com/article/elon-musk-openai-lawsuit-sam-altman-3cd261b2a9b04630ec93582020c59ef7), [Reuters](https://www.reuters.com/legal/elon-musk-openai-head-court-spar-over-nonprofit-conversion-2025-02-04/)).

### How is OpenAI planning to leverage its AI tools for the public sector?

OpenAI aims to leverage its AI tools to address complex public sector challenges and enhance government efficiency. The company's initiatives include the launch of "ChatGPT Gov," a specialized version of their ChatGPT model tailored for U.S. government use. This version is designed to handle sensitive information, support public health, improve infrastructure, and strengthen national security. OpenAI plans to integrate its AI solutions within federal agencies to expedite internal processes and enhance productivity. The tools aim to maintain America's leadership in AI on a global scale. The deployment aligns with OpenAI's broader vision of collaborating with national laboratories and other government institutions to utilize AI for societal benefits ([Digital Information World](https://www.digitalinformationworld.com/2025/01/openai-rolls-out-new-chatgpt-gov-to.html), [American Bazaar](https://americanbazaaronline.com/2025/01/29/openai-launches-chatgpt-gov-ai-chatbot-tailored-for-u-s-government-use458838/)).

### What does OpenAI's new 'Deep Research' tool aim to achieve?

OpenAI's 'Deep Research' tool is designed to conduct thorough, multi-step research on the internet, providing deep, in-depth analyses that rival the output of human research analysts. The tool leverages the capabilities of OpenAI's o3 reasoning model, optimized for web browsing and data analysis, to compile detailed reports quickly—ranging from five to 30 minutes depending on the complexity of the task. Deep Research aims to assist users by automating extensive research tasks, generating comprehensive, fully cited research papers on various topics. This tool is expected to be valuable across industries, pushing the boundaries of what AI can achieve in terms of efficient and precise research work. However, users are cautioned against potential inaccuracies as the tool may sometimes hallucinate facts in its responses ([Geeky Gadgets](https://www.geeky-gadgets.com/openai-deep-research-ai-tool/), [New York Times](https://www.nytimes.com/2025/02/02/technology/openai-deep-research-tool.html)).

In [ ]:
pprint(result.all_messages(),width=160)

[ModelRequest(parts=[SystemPromptPart(content='Given a topic, search for internet news, then generate 4 useful and interesting insights/questions. Answer each '
                                              'question in separate paragraphs, including references.',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='OpenAI in the last two weeks',
                                    timestamp=datetime.datetime(2025, 2, 5, 16, 8, 0, 28619, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='search_news',
                                   args='{"topic": "OpenAI", "date_frame": "{\\"weeks\\":2}"}',
                                   tool_call_id='call_LuZxmMuRFLvzZYsoByU69hTL',
                                   part_kind='tool-call')],
               model_n

In [ ]:
plan_promt = (
                "Given a topic, search for internet news, "
                "then generate 2 useful and interesting insights/questions. "
                "Answer each question in separate paragraphs including 4 link references"
            )
agent_planner = Agent(model=model, result_type=str,
                      system_prompt=plan_promt)
class Serpapi:
    client = serpapi.Client(api_key=userdata.get('SERPAPI_API_KEY'))


    @classmethod
    async def search_topic_until_today(cls, *, question: str, date_frame:str) -> str | None:
        print(f"{question=}, {date_frame=}")
        duckduck_results = cls.client.duckduck_results = client.search({
            'engine': 'duckduckgo',
            'q': question,
            'df': generate_date_frame(**json.loads(date_frame))
        })
        if result:
            context_snippets = list(map(lambda x: f"snippet:{x['snippet']},source:{x['link']}" ,duckduck_results['organic_results']))
            return '\n\n'.join(context_snippets)


@dataclass
class SupportDependencies:
    serpapi_client: Serpapi


@agent_planner.tool
async def search_news(ctx: RunContext[SupportResult], topic: str, date_frame:str) -> str:
    """
    Args:
        date_frame: A limit of days,months,weeks before tday, format in json like {"days":1}, {"months":1}, {"weeks":1}, {"months":1, "weeks":1, "days":1}
    """
    return await ctx.deps.serpapi_client.search_topic_until_today(question=topic, date_frame=date_frame)

@agent_planner.tool
async def answer_an_insight_question(ctx: RunContext[None], insight_question: str) -> str:
    return await ctx.deps.serpapi_client.search_topic_until_today(question=insight_question, date_frame='{"weeks":1}')

deps = SupportDependencies(serpapi_client=Serpapi())
result = agent_planner.run_sync('OpenAI o1', deps=deps, message_history=[])
printmd(result.data)

question='OpenAI', date_frame='{"weeks":1}'
question="What are the main reasons behind OpenAI's decision to shift to a for-profit structure?", date_frame='{"weeks":1}'
question="What are some of the notable achievements or breakthroughs of OpenAI's latest model, o3?", date_frame='{"weeks":1}'


### Insight 1: What are the main reasons behind OpenAI's decision to shift to a for-profit structure?

OpenAI's decision to shift to a for-profit structure is mainly driven by the need to secure substantial funding to continue advancing its research and maintaining a competitive edge in the rapidly evolving AI landscape. This transition allows OpenAI to attract investors by offering potential returns on their investments, which is critical for financing the high costs associated with developing advanced AI technologies. Additionally, the for-profit structure will enable OpenAI to better manage its financial strategy and resource allocation, ensuring sustainable growth and innovation in the long-term. This move is also framed as essential to support its overarching mission of ensuring AGI (Artificial General Intelligence) benefits all of humanity by leveraging the increased capital to enhance their research and operational capabilities. Despite opposition from influential figures such as Elon Musk and organizational critiques, OpenAI emphasizes that transitioning to a for-profit model aligns with its mission by balancing profit motives with public benefit through its incorporation as a Delaware Public Benefit Corporation.

For further details, you can refer to the following sources:
1. OpenAI's official announcement about the for-profit transition: [PCMag](https://www.pcmag.com/news/openai-officially-announces-for-profit-transition)
2. Analyzing the implications for OpenAI’s structure: [The Verge](https://www.theverge.com/2024/12/27/24330131/openai-plan-transform-for-profit-company)
3. Financial strategy and capital requirements: [Reuters](https://www.reuters.com/technology/artificial-intelligence/openai-lays-out-plan-shift-new-for-profit-structure-2024-12-27/)
4. Background and motivation for the shift: [Ars Technica](https://arstechnica.com/tech-policy/2024/12/openai-defends-for-profit-shift-as-critical-to-sustain-humanitarian-mission/)

### Insight 2: What are some of the notable achievements or breakthroughs of OpenAI's latest model, o3?

OpenAI's latest model, o3, has achieved significant milestones, particularly in areas that have traditionally challenged large language models. Notable achievements include a remarkable performance on the ARC-AGI benchmark, where o3 scored 85%, significantly surpassing the previous best score of 55% and aligning closely with the average human score. This level of performance highlights o3's enhanced reasoning and problem-solving capabilities. Additionally, the o3 model has shown significant improvements in specific domains such as coding, where it outperformed earlier models by more than 20% on the SWE-bench Verified benchmark. Another breakthrough is the introduction of "program synthesis," enabling o3 to dynamically combine patterns and algorithms learned during pre-training, which enhances its adaptability and application in varying contexts. These achievements mark a substantial step forward in the development of AI models, suggesting that o3 sets a new standard in general intelligence and practical application across diverse tasks.

For more comprehensive information, see the following references:
1. Overview of o3's achievements and benchmarks: [Gizmodo](https://gizmodo.com/openai-claims-its-new-model-reached-human-level-on-a-test-for-general-intelligence-what-does-that-mean-2000543834)
2. Detailed analysis of o3’s capabilities and breakthroughs: [VentureBeat](https://venturebeat.com/ai/five-breakthroughs-that-make-openais-o3-a-turning-point-for-ai-and-one-big-challenge/)
3. Progress in specific tasks and performance metrics: [Lifehacker](https://lifehacker.com/tech/openai-promises-chatgpt-o3-model-better-at-reasoning)
4. Significance of o3 in the AI research community: [UpThrust](https://upthrust.co/2024/12/what-openais-o3-mode-means-for-the-future-of-artificial-general-intelligence-agi)

# USE HISTORY AND AGENT

In [ ]:
dict_codes = {
    '1': 'John',
    '2': 'Jane',
    '3': 'Bob',
    '4': 'Alice',
    '5': 'Mike',
}

async def get_name_based_on_code(code: str)->str:
    """
        Returns the name of the person with the given code.

    Args:
        - code: The code number of the person.
    """
    return dict_codes[code]

agent = Agent(model=model, result_type=str,
                      system_prompt=(
                          'Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, then'
                          'Use get_name_based_on_code tool to retrieve the name of a person'
                          'Or reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times'
                      ),
              tools=[
                  Tool(get_name_based_on_code),
                     ]
              )
result = agent.run_sync('Tell me the names of persons with code 2 and 5, please')
result

RunResult(_all_messages=[ModelRequest(parts=[SystemPromptPart(content='Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, thenUse get_name_based_on_code tool to retrieve the name of a personOr reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times', dynamic_ref=None, part_kind='system-prompt'), UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 3, 50, 902557, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'), ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 2 and 5:\n- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 2.\n- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 5.\n\nSince both tools can be called simultaneously, I will make parallel requests.\n

In [ ]:
pprint(result.new_messages(),width=220)

[ModelRequest(parts=[SystemPromptPart(content='Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, thenUse get_name_based_on_code tool to retrieve the name of a '
                                              'personOr reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 3, 50, 902557, tzinfo=datetime.timezone.utc), part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 2 and 5:\n"
                                       '- Use the `get_name_based_on_code` tool to retrieve the name of the person with 

In [ ]:
pprint(result.all_messages(),width=220)

[ModelRequest(parts=[SystemPromptPart(content='Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, thenUse get_name_based_on_code tool to retrieve the name of a '
                                              'personOr reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 3, 50, 902557, tzinfo=datetime.timezone.utc), part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 2 and 5:\n"
                                       '- Use the `get_name_based_on_code` tool to retrieve the name of the person with 

In [ ]:
result2 = agent.run_sync('Tell me the names of persons with code 2 and 5, please', message_history=result.all_messages())
result2

RunResult(_all_messages=[ModelRequest(parts=[SystemPromptPart(content='Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, thenUse get_name_based_on_code tool to retrieve the name of a personOr reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times', dynamic_ref=None, part_kind='system-prompt'), UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 3, 50, 902557, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'), ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 2 and 5:\n- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 2.\n- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 5.\n\nSince both tools can be called simultaneously, I will make parallel requests.\n

In [ ]:
pprint(result2.new_messages(),width=220)

[ModelRequest(parts=[UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 4, 25, 669468, tzinfo=datetime.timezone.utc), part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[TextPart(content='The names of the persons with the given codes are:\n- Code 2: Jane\n- Code 5: Mike', part_kind='text')],
               model_name='gpt-4o',
               timestamp=datetime.datetime(2025, 2, 5, 17, 4, 25, tzinfo=datetime.timezone.utc),
               kind='response')]


In [ ]:
result3 = agent.run_sync('Tell me the names of persons with code 1 and 5, please', message_history=result.all_messages())
result3

RunResult(_all_messages=[ModelRequest(parts=[SystemPromptPart(content='Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, thenUse get_name_based_on_code tool to retrieve the name of a personOr reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times', dynamic_ref=None, part_kind='system-prompt'), UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 3, 50, 902557, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'), ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 2 and 5:\n- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 2.\n- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 5.\n\nSince both tools can be called simultaneously, I will make parallel requests.\n

In [ ]:
pprint(result3.new_messages(),width=220)

[ModelRequest(parts=[UserPromptPart(content='Tell me the names of persons with code 1 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 4, 32, 674119, tzinfo=datetime.timezone.utc), part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 1 and 5:\n"
                                       '- Use the `get_name_based_on_code` tool to retrieve the name of the person with code 1.\n'
                                       '- Reuse the previously retrieved result for code 5 (Mike), as we already have this information.\n'
                                       '\n'
                                       "Let's proceed with that.",
                               part_kind='text'),
                      ToolCallPart(tool_name='get_name_based_on_code', args='{"code":"1"}', tool_call_id='call_pQsMPAT8nbyU7aEbwNMZyuZC', part_kind='tool-call')],
               model_name='gpt-4o',
         

In [ ]:
pprint(result3.all_messages(),width=220)

[ModelRequest(parts=[SystemPromptPart(content='Generate a plan message with bullet points(DO NOT FORGET) where indicates the steps of tools to call, thenUse get_name_based_on_code tool to retrieve the name of a '
                                              'personOr reuse results if the question was answered to avoid using the tool result when it has been used/responsed tool multiple times',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Tell me the names of persons with code 2 and 5, please', timestamp=datetime.datetime(2025, 2, 5, 17, 3, 50, 902557, tzinfo=datetime.timezone.utc), part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[TextPart(content="Here's the plan to retrieve the names of persons with code 2 and 5:\n"
                                       '- Use the `get_name_based_on_code` tool to retrieve the name of the person with 